# CEO Cost & Margin Dashboard — D2C Fashion

**⚠️ ALL DATA IN THIS NOTEBOOK IS SYNTHETIC.** It is generated by a seeded
random data generator to look like a plausible small D2C fashion brand. It
is not real sales, marketing, or inventory data, and no business conclusion
should be drawn from the specific numbers — only from the *methodology*.

## What this notebook is

A two-layer analytics engine for a goods-selling business's cost and margin
performance, built so a CEO can see revenue, margin, working capital,
returns, acquisition efficiency, and inventory health in one place — with
every number traceable to a plain-language formula.

**Layer 1 — Universal Core**: KPIs that apply to *any* goods business
(revenue, gross margin, contribution margin, COGS/opex breakdown,
inventory turns, working-capital cycle, SKU concentration).

**Layer 2 — Fashion Module**: KPIs specific to D2C fashion (returns rate,
CAC/ROAS/MER, LTV:CAC, AOV, repeat rate, cohort retention, sell-through,
weeks-of-cover, markdown/dead-stock %), defined against the **same** core
engine via a config file rather than by editing the engine itself. Swapping
industries (e.g. to FMCG or auto parts) is meant to mean *writing a new
config*, not touching Layer 1's calculation code.

## No black boxes

Every KPI's formula and the business question it answers is written out in
a markdown cell before it's used. If a line of code in this notebook can't
be explained in plain language in an interview, it doesn't belong here.

## Build stages (this notebook is built incrementally)

1. **Data generator + swappable loader** ← this notebook, current stage
2. Cleaning + Universal Core KPIs
3. Fashion industry module (returns, CAC, cohorts, sell-through)
4. Dashboard / visualizations
5. Margin-risk alert model
6. Demand / inventory forecast model

---
## Stage 1 — Synthetic data generator and the swappable column-mapping loader

**Goal of this stage:** produce a realistic order-level D2C fashion dataset
and prove that swapping in a real CSV later requires editing only a
column-mapping dictionary — never the analysis code.

### Setup — installs (Colab) and imports

In [1]:
# Colab: uncomment the line below on first run in a fresh Colab environment.
# !pip install -q pandas numpy matplotlib plotly scikit-learn

import sys
sys.path.append("..")  # so `from src import ...` resolves when running from notebooks/

import pandas as pd
import numpy as np

from src import schema
from src import data_generator as dg
from src import data_loader as dl

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

### Why three tables, not one giant spreadsheet

A real D2C business's data doesn't naturally live in a single flat table,
and forcing it into one would hide real structure:

| Table | Grain | Why it's separate |
|---|---|---|
| `orders` | one row per **order line item** | The core fact table: revenue, cost, returns, and customer identity all live here. |
| `marketing_spend` | one row per **(month, channel)** | Ad spend is a channel-level input (an ad platform invoice) — it isn't a property of one order, so attaching a "spend" column to every order line would double- or triple-count it the moment an order has multiple line items. |
| `inventory_snapshots` | one row per **(month, SKU)** | Turns, days-of-inventory, and sell-through all need what *wasn't* sold (opening/closing stock), which the orders table alone can never tell you. |

This mirrors how the data actually arrives from a real stack (Shopify/POS
export, ad-platform report, WMS export) — three separate files that get
joined for analysis, not reconciled from a single source.

### Generating the synthetic dataset

`src/data_generator.py` builds all three tables from a single seeded
`numpy.random.Generator` (default seed 42), so the dataset is 100%
reproducible — rerunning this cell (or the whole notebook) on any machine
produces byte-identical output.

Key generation choices, made explicit here so nothing downstream is a
surprise:
- **12 months** of order-line data (Jul 2025 – Jun 2026), 6 product
  categories, ~150 SKUs, an order-volume ramp (the brand is growing) and two
  seasonal sale-driven demand spikes.
- **Return rates are category-specific and deliberately high for
  fit-sensitive categories** (Dresses ~38%, Tops ~32%, Bottoms ~28%),
  landing in the 20–40% range real fashion brands see, specifically so the
  margin impact of returns is visible rather than a rounding error.
- **Customers arrive over time and a share of them re-order** later in the
  year, which is what makes repeat-purchase-rate and cohort retention
  (Stage 3) meaningful rather than trivially zero.
- **Marketing spend per channel roughly tracks the orders that channel is
  credited with**, plus noise and a channel-specific efficiency factor —
  so CAC/ROAS differ meaningfully by channel instead of being flat by
  construction.
- **Inventory purchasing is deliberately imperfect for a subset of SKUs**
  (some over-bought, some under-bought) so later stages have genuine
  overstock/stockout cases to detect.

Full assumptions are documented in the `data_generator.py` module
docstring.

In [2]:
data = dg.generate_all(seed=42)
orders_df = data["orders"]
marketing_df = data["marketing_spend"]
inventory_df = data["inventory_snapshots"]

print("SYNTHETIC DATA — generated with seed=42, fully reproducible\n")
for name, df in data.items():
    print(f"{name:22s} shape={df.shape}")

SYNTHETIC DATA — generated with seed=42, fully reproducible

orders                 shape=(25930, 15)
marketing_spend        shape=(60, 3)
inventory_snapshots    shape=(1651, 6)
opex                   shape=(60, 3)


### Orders table — schema and sample

In [3]:
print("orders.csv column dtypes:\n")
print(orders_df.dtypes)
orders_df.head(5)

orders.csv column dtypes:

order_id                          str
order_line_id                     str
order_date             datetime64[us]
customer_id                       str
sku_id                            str
category                          str
size                              str
quantity                        int64
unit_price                    float64
unit_cogs                     float64
shipping_cost                 float64
payment_gateway_fee           float64
marketing_channel                 str
is_return                        bool
return_date            datetime64[us]
dtype: object


,order_id,order_line_id,order_date,customer_id,sku_id,category,size,quantity,unit_price,unit_cogs,shipping_cost,payment_gateway_fee,marketing_channel,is_return,return_date
0,ORD-000001,ORD-000001-1,2025-07-01,CUST-00001,BOT-025,Bottoms,M,1,1859.0,754.26,49.15,41.04,Organic/Email,True,2025-07-10
1,ORD-000015,ORD-000015-1,2025-07-01,CUST-00014,ACC-009,Accessories,One Size,1,1119.0,390.44,62.30,25.50,Affiliate,False,NaT
2,ORD-000016,ORD-000016-1,2025-07-01,CUST-00015,TOP-023,Tops,M,1,879.0,300.27,77.87,20.46,Instagram Ads,False,NaT
3,ORD-000017,ORD-000017-1,2025-07-01,CUST-00007,OUT-007,Outerwear,M,1,2649.0,1026.21,72.83,57.63,Organic/Email,False,NaT
4,ORD-000018,ORD-000018-1,2025-07-01,CUST-00016,FOO-016,Footwear,11,2,3639.0,1557.89,91.71,154.84,Influencer,False,NaT


**Column-by-column explanation of `orders`** (one row = one order line item):

| Column | Meaning |
|---|---|
| `order_id` | Groups line items placed in the same checkout — an order with 2 SKUs is 2 rows sharing this value. |
| `order_line_id` | Unique key for this row (`order_id` + line number). |
| `order_date` | Date the order was placed. |
| `customer_id` | Identifies the buyer; the same value recurs across a customer's repeat orders — this is what makes repeat-rate and cohort analysis possible. |
| `sku_id` | Product+variant identifier (e.g. `DRE-014`). |
| `category` | Product category — Tops, Bottoms, Dresses, Outerwear, Footwear, or Accessories. |
| `size` | Size variant of the SKU. |
| `quantity` | Units on this line (mostly 1, occasionally 2). |
| `unit_price` | Realized selling price per unit, **after** any discount — this is what actually hit revenue, not the list price. |
| `unit_cogs` | Cost of goods per unit for this SKU (manufacturing/sourcing cost). |
| `shipping_cost` | Fulfillment/shipping cost allocated to this line. |
| `payment_gateway_fee` | Payment processor fee allocated to this line (~2.1% + a small fixed fee, typical of Indian payment gateways). |
| `marketing_channel` | The channel credited with driving the *parent order* (Instagram Ads, Google Ads, Influencer, Affiliate, or Organic/Email). |
| `is_return` | `True` if this specific line item was returned. |
| `return_date` | Date of the return; `NaT` (missing) if never returned. |

### Marketing spend table — schema and sample

In [4]:
print("marketing_spend.csv column dtypes:\n")
print(marketing_df.dtypes)
marketing_df.head(5)

marketing_spend.csv column dtypes:

month          str
channel        str
spend      float64
dtype: object


,month,channel,spend
0,2025-07,Affiliate,22688.01
1,2025-07,Google Ads,96084.07
2,2025-07,Influencer,62745.79
3,2025-07,Instagram Ads,83227.15
4,2025-07,Organic/Email,0.00


**Column-by-column explanation of `marketing_spend`** (one row = one
channel's total spend in one month):

| Column | Meaning |
|---|---|
| `month` | Calendar month, `YYYY-MM`. |
| `channel` | Marketing channel (matches `orders.marketing_channel`). |
| `spend` | Total spend on that channel that month. `Organic/Email` is always 0 — it represents word-of-mouth, direct, and retention email, not a paid acquisition channel. |

This table is joined to `orders` (aggregated by month + channel) when
computing CAC/ROAS in Stage 3 — it is never merged row-for-row onto order
lines, because that would silently duplicate spend across every line item
of a multi-item order.

### Inventory snapshots table — schema and sample

In [5]:
print("inventory_snapshots.csv column dtypes:\n")
print(inventory_df.dtypes)
inventory_df.head(5)

inventory_snapshots.csv column dtypes:

sku_id                   str
month                    str
beginning_inventory    int64
units_received         int64
units_sold             int64
ending_inventory       int64
dtype: object


,sku_id,month,beginning_inventory,units_received,units_sold,ending_inventory
0,ACC-001,2025-09,26,27,11,42
1,ACC-001,2025-10,42,0,6,36
2,ACC-001,2025-11,36,0,28,8
3,ACC-001,2025-12,8,25,17,16
4,ACC-001,2026-01,16,25,20,21


**Column-by-column explanation of `inventory_snapshots`** (one row = one
SKU's stock position in one month):

| Column | Meaning |
|---|---|
| `sku_id` | Matches `orders.sku_id`. |
| `month` | Calendar month, `YYYY-MM`. |
| `beginning_inventory` | Units on hand at the start of the month. |
| `units_received` | Units restocked during the month. |
| `units_sold` | Units sold during the month (gross, before returns) — reconciles with `orders` grouped by SKU and month. |
| `ending_inventory` | `beginning_inventory + units_received - units_sold`, floored at 0. Next month's `beginning_inventory` always equals this value — verified in the test suite. |

### Quick sanity check: does this data look like a real fashion brand?

In [6]:
total_revenue = (orders_df["unit_price"] * orders_df["quantity"]).sum()
print(f"Date range        : {orders_df['order_date'].min().date()} to {orders_df['order_date'].max().date()}")
print(f"Unique orders      : {orders_df['order_id'].nunique():,}")
print(f"Unique customers   : {orders_df['customer_id'].nunique():,}")
print(f"Unique SKUs        : {orders_df['sku_id'].nunique()}")
print(f"Order line items   : {len(orders_df):,}")
print(f"Gross revenue (pre-return, INR): {total_revenue:,.0f}")
print()
print("Return rate by category (target: fit-sensitive categories land in the realistic 20-40% fashion range):")
print(orders_df.groupby("category")["is_return"].mean().sort_values(ascending=False).round(3))

Date range        : 2025-07-01 to 2026-06-30
Unique orders      : 18,607
Unique customers   : 12,002
Unique SKUs        : 150
Order line items   : 25,930
Gross revenue (pre-return, INR): 51,692,051

Return rate by category (target: fit-sensitive categories land in the realistic 20-40% fashion range):
category
Dresses        0.377
Tops           0.327
Bottoms        0.285
Footwear       0.212
Outerwear      0.171
Accessories    0.088
Name: is_return, dtype: float64


Dresses, Tops, and Bottoms — the fit-sensitive categories — land in the
28–38% return-rate range, Accessories sits under 10%. This spread is by
design (see the generator's category return-rate table) and is what will
make the returns-driven margin impact visible in Stage 3, rather than a
number too small to matter.

### The swappable loader — proving a real CSV export could replace this data with zero analysis-code changes

In [7]:
import os

# Save the generated tables exactly as a real export would arrive: plain CSVs.
os.makedirs("../data/synthetic", exist_ok=True)
dg.save_all(data, "../data/synthetic")
print("Saved:", os.listdir("../data/synthetic"))

Saved: ['inventory_snapshots.csv', 'opex.csv', 'marketing_spend.csv', 'orders.csv']


`src/data_loader.py` reads these CSVs and renames columns using a mapping
dict (`schema.DEFAULT_ORDERS_MAPPING`, etc.) — by default an identity
mapping, since the synthetic data already uses canonical column names. To
prove this is a real swap point (not just a pass-through), the cell below
simulates a real export with **different column headers** — `"Order No"`
instead of `order_id`, `"Selling Price"` instead of `unit_price` — and
loads it by editing only a mapping dict, changing no code in
`data_loader.py` or any calculation module.

In [8]:
# Simulate a real export with different headers than our canonical schema.
fake_export = pd.read_csv("../data/synthetic/orders.csv")
fake_export = fake_export.rename(columns={"order_id": "Order No", "unit_price": "Selling Price"})
fake_export.to_csv("../data/synthetic/_demo_real_export.csv", index=False)

# Build a custom mapping: copy the default, then repoint the two changed headers.
custom_mapping = dict(schema.DEFAULT_ORDERS_MAPPING)
del custom_mapping["order_id"]
del custom_mapping["unit_price"]
custom_mapping["Order No"] = schema.ORDER_ID
custom_mapping["Selling Price"] = schema.UNIT_PRICE

swapped_df = dl.load_orders("../data/synthetic/_demo_real_export.csv", mapping=custom_mapping)
print("Loaded a differently-headed file using only a mapping-dict edit.")
print("Resulting columns match the canonical schema:", list(swapped_df.columns) == schema.ORDERS_SCHEMA)
swapped_df.head(3)

Loaded a differently-headed file using only a mapping-dict edit.
Resulting columns match the canonical schema: True


,order_id,order_line_id,order_date,customer_id,sku_id,category,size,quantity,unit_price,unit_cogs,shipping_cost,payment_gateway_fee,marketing_channel,is_return,return_date
0,ORD-000001,ORD-000001-1,2025-07-01,CUST-00001,BOT-025,Bottoms,M,1,1859.0,754.26,49.15,41.04,Organic/Email,True,2025-07-10
1,ORD-000015,ORD-000015-1,2025-07-01,CUST-00014,ACC-009,Accessories,One Size,1,1119.0,390.44,62.30,25.50,Affiliate,False,NaT
2,ORD-000016,ORD-000016-1,2025-07-01,CUST-00015,TOP-023,Tops,M,1,879.0,300.27,77.87,20.46,Instagram Ads,False,NaT


In [9]:
os.remove("../data/synthetic/_demo_real_export.csv")  # clean up the demo file

If a real export is missing a column the engine needs, the loader raises
immediately with the missing column's name (`schema.validate_columns`),
rather than letting a KPI compute silently on absent data. This is checked
in `tests/test_stage1_data.py::test_loader_raises_on_missing_column`.

---
## Stage 1 summary

Built a fully synthetic, seeded, reproducible 12-month D2C fashion dataset
across three tables — `orders` (order-line grain, 25,930 lines / ~18,600
orders / 12,002 customers / 150 SKUs), `marketing_spend` (channel × month),
and `inventory_snapshots` (SKU × month) — with category-specific return
rates realistically landing in the 20–40% fashion range (Dresses ~38%,
Tops ~32%) so their margin impact will be visible later. Every column is
documented above. The data-loading layer (`src/data_loader.py` +
`src/schema.py`) was proven swappable: a differently-headed CSV loads
correctly by editing only a column-mapping dictionary, with no changes to
loader or analysis code, and a missing required column fails loudly instead
of silently. 9 automated tests cover schema shape, reproducibility,
inventory-chain consistency, and the loader swap/failure paths (all
passing).

**Stopping here for review before Stage 2 (cleaning + Universal Core KPIs).**

---
## Stage 2 — Cleaning + Universal Core KPIs (Layer 1)

Two things happen in this stage:

1. **Cleaning** (`src/clean.py`): a small, logged pipeline that would catch
   the mess a real raw export has (duplicates, missing costs, inconsistent
   text, invalid values) -- demonstrated against a deliberately messy
   sample, then run for real against the canonical dataset.
2. **Universal Core KPIs** (`src/core.py`): revenue, margin, cost
   breakdowns, inventory efficiency, and working capital -- every formula
   works for *any* goods business. Nothing fashion-specific appears in
   `core.py`; that boundary is enforced by construction, not just by
   convention.

### Pre-flight diagnostic (before writing any KPI code)

Before building the core layer, two questions needed real numbers, not
assumptions:

1. **Does this dataset actually contain loss-making SKUs and channels?** A
   margin-risk alert model (Stage 5) is pointless to build against data
   where nothing is ever unprofitable.
2. **How is marketing spend currently related to CAC?** Is it already
   *true* CAC (spend ÷ newly acquired customers), or just a cost-per-order
   figure? This matters because Stage 3/5 will build real CAC on top of
   this data, and the difference changes every number downstream.

This diagnostic uses one-off code, not `core.py` functions -- CAC and
marketing-loaded margin are Layer 2 (fashion/channel) concepts and
deliberately do not belong in the universal core engine. The proper,
reusable versions get built in Stage 3.

In [10]:
import pandas as pd
import numpy as np

from src import data_generator as dg
from src import clean
from src import core
from src import schema

data = dg.generate_all(seed=42)
cleaned_preview, _ = clean.clean_orders(data["orders"])
econ_preview = core.add_line_economics(cleaned_preview)
marketing_df = data["marketing_spend"]

# ---- Q1: cost-per-order vs true CAC (spend / newly acquired customers) ----
orders_only = econ_preview.drop_duplicates(subset=schema.ORDER_ID).copy()
first_orders = (orders_only.sort_values(schema.ORDER_DATE)
                 .drop_duplicates(subset=schema.CUSTOMER_ID, keep="first"))

orders_by_channel = orders_only.groupby(schema.MARKETING_CHANNEL).size()
new_cust_by_channel = first_orders.groupby(schema.MARKETING_CHANNEL).size()
spend_by_channel = marketing_df.groupby(schema.SPEND_CHANNEL)[schema.SPEND_AMOUNT].sum()

cac_check = pd.DataFrame({
    "total_spend": spend_by_channel,
    "total_orders_credited": orders_by_channel,
    "total_new_customers": new_cust_by_channel,
}).fillna(0)
cac_check["cost_per_order"] = cac_check["total_spend"] / cac_check["total_orders_credited"]
cac_check["true_cac_per_new_customer"] = (
    cac_check["total_spend"].replace(0, np.nan) / cac_check["total_new_customers"].replace(0, np.nan)
)
print("Q1 -- cost-per-order vs true CAC by channel:")
print(cac_check.round(2).to_string())
print()
print("The generator builds marketing_spend as (a per-order cost rate) x (ALL orders credited to")
print("the channel, new + repeat). So spend / orders_credited recovers a COST-PER-ORDER, not true")
print("CAC. True CAC (spend / new customers only) is 20-30% higher for every paid channel, since")
print("spend also covers repeat-customer orders attributed to that channel, which the new-customer")
print("count excludes. Stage 3's CAC KPI will use the TRUE definition (spend / new customers).")

Q1 -- cost-per-order vs true CAC by channel:
               total_spend  total_orders_credited  total_new_customers  cost_per_order  true_cac_per_new_customer
Affiliate        356291.68                   1505                 1176          236.74                     302.97
Google Ads      1320539.64                   3226                 2899          409.34                     455.52
Influencer      1073898.13                   2147                 1864          500.19                     576.13
Instagram Ads   1749176.48                   4505                 4128          388.27                     423.73
Organic/Email         0.00                   7224                 1935            0.00                        NaN

The generator builds marketing_spend as (a per-order cost rate) x (ALL orders credited to
the channel, new + repeat). So spend / orders_credited recovers a COST-PER-ORDER, not true
CAC. True CAC (spend / new customers only) is 20-30% higher for every paid channel, since

In [11]:
# ---- Q2: how many SKUs / channels are loss-making after returns + CAC? ----

# Channel-level: contribution margin generated by orders attributed to a
# channel vs. total spend on that channel (a channel "mini P&L").
cm_by_channel = econ_preview.groupby(schema.MARKETING_CHANNEL)["contribution_margin_line"].sum()
channel_pnl = pd.DataFrame({
    "contribution_margin": cm_by_channel,
    "orders": orders_by_channel,
    "spend": spend_by_channel,
}).fillna(0)
channel_pnl["cac_cost_per_order"] = channel_pnl["spend"] / channel_pnl["orders"]
channel_pnl["cm_after_cac"] = channel_pnl["contribution_margin"] - channel_pnl["spend"]
n_neg_channels = int((channel_pnl["cm_after_cac"] < 0).sum())

print("Q2a -- channel P&L (contribution margin generated minus spend on that channel):")
print(channel_pnl.round(2).to_string())
print(f"\nChannels with NEGATIVE contribution margin after CAC: {n_neg_channels} of {len(channel_pnl)}")

# SKU-level: allocate each channel's per-order spend pro-rata across that
# order's lines by revenue share, down to SKU grain.
order_net_rev = econ_preview.groupby(schema.ORDER_ID)["net_revenue_line"].transform("sum")
line_count = econ_preview.groupby(schema.ORDER_ID)[schema.ORDER_ID].transform("count")
econ_preview["line_share"] = np.where(order_net_rev > 0, econ_preview["net_revenue_line"] / order_net_rev, 1 / line_count)
econ_preview["allocated_cac"] = (
    econ_preview[schema.MARKETING_CHANNEL].map(channel_pnl["cac_cost_per_order"]) * econ_preview["line_share"]
)
econ_preview["cm_after_cac_line"] = econ_preview["contribution_margin_line"] - econ_preview["allocated_cac"]

sku_cac = econ_preview.groupby(schema.SKU_ID).agg(
    contribution_margin=("contribution_margin_line", "sum"),
    cm_after_cac=("cm_after_cac_line", "sum"),
).reset_index()
n_neg_sku_before_cac = int((sku_cac["contribution_margin"] < 0).sum())
n_neg_sku_after_cac = int((sku_cac["cm_after_cac"] < 0).sum())

print()
print(f"Q2b -- SKUs with negative contribution margin BEFORE CAC (returns already netted in): {n_neg_sku_before_cac} of {len(sku_cac)}")
print(f"Q2b -- SKUs with negative contribution margin AFTER pro-rata CAC allocation: {n_neg_sku_after_cac} of {len(sku_cac)}")
print()
print("Bottom 6 SKUs by contribution margin after CAC:")
print(sku_cac.sort_values("cm_after_cac").head(6).round(2).to_string(index=False))

Q2a -- channel P&L (contribution margin generated minus spend on that channel):
               contribution_margin  orders       spend  cac_cost_per_order  cm_after_cac
Affiliate               1531716.52    1505   356291.68              236.74    1175424.84
Google Ads              3485934.40    3226  1320539.64              409.34    2165394.76
Influencer              2346533.35    2147  1073898.13              500.19    1272635.22
Instagram Ads           4585477.04    4505  1749176.48              388.27    2836300.56
Organic/Email           7353649.62    7224        0.00                0.00    7353649.62

Channels with NEGATIVE contribution margin after CAC: 0 of 5

Q2b -- SKUs with negative contribution margin BEFORE CAC (returns already netted in): 0 of 150
Q2b -- SKUs with negative contribution margin AFTER pro-rata CAC allocation: 3 of 150

Bottom 6 SKUs by contribution margin after CAC:
 sku_id  contribution_margin  cm_after_cac
ACC-010             15241.64     -13907.68
TOP-028

**Findings, stated plainly:**

- **0 of 5 channels** show negative contribution margin after CAC. Even
  Influencer, the least efficient paid channel, still clears its own
  acquisition cost by a wide margin (contribution margin per order ≈ ₹1,093
  vs. CAC-implied cost per order ≈ ₹500).
- **0 of 150 SKUs** are loss-making before CAC (returns already netted
  in); only **3 of 150** turn negative after a pro-rata CAC allocation, and
  a handful more sit within a few hundred rupees of zero.
- **Flagging this explicitly, as requested**: this signal is too thin. A
  margin-risk alert model (Stage 5) trained or ruled against data where
  almost nothing is ever unprofitable will have nothing real to catch. The
  category economics (60%+ gross margin everywhere) and channel CAC
  (modest relative to a ~₹1,900 AOV) are healthy by construction, with no
  deliberately weak SKUs or channels seeded in Stage 1.
- **This is a data-generation calibration issue, not a Stage 2 problem.**
  Universal Core KPIs (below) are correct regardless of how many things
  are profitable. The fix -- seeding a small number of genuinely
  loss-making SKUs (e.g. high-return, thin-margin items) and one
  under-performing channel -- belongs in `src/data_generator.py`, and is
  deferred to just before Stage 5 rather than reopening the already-
  reviewed Stage 1 dataset here without cause.

### Cleaning pipeline

`src/clean.py` runs four named steps, each logging what it found and what
policy it applied -- nothing is silently imputed or silently dropped:

| Step | What it catches | Policy |
|---|---|---|
| `normalise_categories` | inconsistent casing/whitespace (e.g. `"TOPS  "`) | strip + title-case |
| `deduplicate_order_lines` | exact duplicate `order_line_id` (e.g. a webhook retry) | drop, keep first |
| `flag_missing_cogs` | null `unit_cogs` | **never imputed** -- row kept, flagged `valid_economics=False`, excluded from every cost/margin KPI |
| `flag_invalid_economics` | `quantity<=0` or `unit_price<=0` (a data bug, not a real sale) | **never coerced** -- same flag/exclude policy |

To prove this pipeline does real work (the canonical Stage 1 dataset is
clean by construction), the next cell runs it against a deliberately
messy copy first -- `src/raw_noise.py` injects duplicate rows, missing
costs, casing issues, and invalid values at realistic small rates,
simulating what a real raw export actually looks like.

In [12]:
from src import raw_noise

rng = np.random.default_rng(123)
messy_orders = raw_noise.make_messy(data["orders"], rng)
print(f"Messy sample: {len(messy_orders):,} rows (vs {len(data['orders']):,} clean rows -- duplicates added)")

cleaned_messy, decision_log = clean.clean_orders(messy_orders)
print("\nDecision log (cleaning the messy sample):")
print(decision_log.to_string(index=False))
print(f"\nRows flagged valid_economics=False: {(~cleaned_messy['valid_economics']).sum()} of {len(cleaned_messy)}")

Messy sample: 26,033 rows (vs 25,930 clean rows -- duplicates added)

Decision log (cleaning the messy sample):
                   step  rows_affected                                                                                                                              rule_applied
   normalise_categories            520                                                                                                              strip whitespace, title-case
deduplicate_order_lines            103                                                                                 drop exact duplicate order_line_id, keep first occurrence
      flag_missing_cogs            130 unit_cogs is never imputed; rows flagged valid_economics=False and excluded from every cost/margin KPI, kept for revenue/return reporting
 flag_invalid_economics             52                                 quantity<=0 or unit_price<=0 is not a real sale; rows flagged valid_economics=False, values never coerced

Ro

Now the real thing: cleaning the actual canonical dataset. Because it was
generated clean, every step should report **zero** rows affected -- which
is itself worth confirming, not assuming.

In [13]:
orders_clean, decision_log_real = clean.clean_orders(data["orders"])
print("Decision log (cleaning the real canonical orders table):")
print(decision_log_real.to_string(index=False))
assert (decision_log_real["rows_affected"] == 0).all(), "canonical data should be clean by construction"
print("\nConfirmed: canonical data is clean by construction, as expected.")
print(f"orders_clean shape: {orders_clean.shape} (includes new 'valid_economics' column)")

opex_df = data["opex"]
inventory_df = data["inventory_snapshots"]

Decision log (cleaning the real canonical orders table):
                   step  rows_affected                                                                                                                              rule_applied
   normalise_categories              0                                                                                                              strip whitespace, title-case
deduplicate_order_lines              0                                                                                 drop exact duplicate order_line_id, keep first occurrence
      flag_missing_cogs              0 unit_cogs is never imputed; rows flagged valid_economics=False and excluded from every cost/margin KPI, kept for revenue/return reporting
 flag_invalid_economics              0                                 quantity<=0 or unit_price<=0 is not a real sale; rows flagged valid_economics=False, values never coerced

Confirmed: canonical data is clean by construction, as ex

From here on, **`orders_clean`** (the output of `clean.clean_orders`) is
the input every Universal Core function expects. Every function below
calls `core.add_line_economics` first internally, which both filters to
`valid_economics == True` rows and derives the per-line revenue/cost
columns every KPI is built from -- see its docstring in `src/core.py` for
the exact return-handling model (returns zero out revenue and COGS, but
NOT shipping/payment fees, which is what makes returns a real margin
drag).

### KPI 1 — Revenue trend

**Formula:** `gross_revenue` = Σ(unit_price × quantity) over all lines;
`net_revenue` = the same, excluding returned lines; `returned_revenue` =
gross − net.
**CEO question:** is the top line growing, and how much of it is being
given back through returns each month?

In [14]:
revenue_trend = core.revenue_trend(orders_clean)
revenue_trend

,month,gross_revenue,net_revenue,returned_revenue,return_rate_value
0,2025-07,2751618.22,2020627.12,730991.10,0.265659
1,2025-08,3413734.12,2584162.68,829571.44,0.243010
2,2025-09,3484406.78,2560232.59,924174.19,0.265231
3,2025-10,3603263.29,2693371.26,909892.03,0.252519
4,2025-11,4224153.72,3145650.16,1078503.56,0.255318
5,2025-12,3824690.48,2890626.95,934063.53,0.244219
6,2026-01,4542603.32,3395291.28,1147312.04,0.252567
7,2026-02,4268325.46,3033763.62,1234561.84,0.289238
8,2026-03,4643416.79,3488262.66,1155154.13,0.248772
9,2026-04,4663836.29,3324136.32,1339699.97,0.287253


### KPI 2 — Gross Margin

**Formula:** Gross Margin % = (Net Revenue − Net COGS) / Net Revenue.
**CEO question:** is the core product economics (price vs. cost to
make/source it) healthy, independent of fulfillment, marketing, or
overhead cost?

In [15]:
gross_margin_trend = core.gross_margin_trend(orders_clean)
gross_margin_trend

,month,net_revenue,net_cogs,gross_margin_abs,gross_margin_pct
0,2025-07,2020627.12,789235.35,1231391.77,0.609411
1,2025-08,2584162.68,1000735.67,1583427.01,0.612743
2,2025-09,2560232.59,996489.68,1563742.91,0.610782
3,2025-10,2693371.26,1053265.42,1640105.84,0.608942
4,2025-11,3145650.16,1669244.73,1476405.43,0.469348
5,2025-12,2890626.95,1125115.00,1765511.95,0.610771
6,2026-01,3395291.28,1327743.36,2067547.92,0.608946
7,2026-02,3033763.62,1171757.20,1862006.42,0.613761
8,2026-03,3488262.66,1348161.13,2140101.53,0.613515
9,2026-04,3324136.32,1280040.85,2044095.47,0.614925


### KPI 3 — Contribution Margin (absolute and per order)

**Formula:** Contribution Margin = Net Revenue − Net COGS − Variable
Fulfillment Costs (shipping + payment gateway fees). Contribution Margin
per Order = Contribution Margin ÷ number of distinct orders that month.
**CEO question:** after making/sourcing the product AND fulfilling the
order, how much is left to cover marketing and overhead -- per rupee of
revenue and per order taken? (Marketing-loaded economics is a Layer 2
concern, built on top of this in Stage 3/5.)

Returns are already netted into Net Revenue/Net COGS above, but that
hides exactly how much margin they cost. The bridge below makes it a
visible, separate line.

In [16]:
contribution_margin_trend = core.contribution_margin_trend(orders_clean)
contribution_margin_trend

,month,contribution_margin_abs,n_orders,contribution_margin_per_order
0,2025-07,1083812.31,894,1212.318020
1,2025-08,1396755.75,1122,1244.880348
2,2025-09,1376669.65,1140,1207.604956
3,2025-10,1445376.67,1177,1228.017562
4,2025-11,1195485.97,1926,620.709226
5,2025-12,1555035.64,1294,1201.727697
6,2026-01,1821465.06,1501,1213.501039
7,2026-02,1628824.54,1442,1129.559320
8,2026-03,1882016.46,1608,1170.408246
9,2026-04,1787708.48,1608,1111.759005


In [17]:
returns_bridge = core.returns_margin_bridge(orders_clean)
print("Returns margin bridge (gross_contribution_margin = as if nothing were ever returned):")
print(returns_bridge.round(0).to_string(index=False))

total_impact = returns_bridge["returns_margin_impact"].sum()
total_gross_cm = returns_bridge["gross_contribution_margin"].sum()
print(f"\nOver 12 months, returns cost ₹{-total_impact:,.0f} of contribution margin")
print(f"-- {-total_impact/total_gross_cm:.1%} of what contribution margin would have been with no returns at all.")

Returns margin bridge (gross_contribution_margin = as if nothing were ever returned):
  month  gross_contribution_margin  returns_margin_impact  net_contribution_margin
2025-07                  1534818.0              -451006.0                1083812.0
2025-08                  1905370.0              -508614.0                1396756.0
2025-09                  1946949.0              -570280.0                1376670.0
2025-10                  2008146.0              -562769.0                1445377.0
2025-11                  1707003.0              -511517.0                1195486.0
2025-12                  2131943.0              -576907.0                1555036.0
2026-01                  2530033.0              -708568.0                1821465.0
2026-02                  2393027.0              -764202.0                1628825.0
2026-03                  2592983.0              -710967.0                1882016.0
2026-04                  2621387.0              -833678.0                1787708.0
2

**Returns wipe out roughly 29% of gross contribution margin over the
year** -- this is the single clearest argument in the whole dashboard for
treating returns as a first-class margin driver, not a customer-service
footnote. Stage 3 breaks this down by category and SKU.

### KPI 4 — COGS breakdown and Operating Expense breakdown

**Formula (COGS):** Σ(net_cogs_line) grouped by category, as a % of total
COGS. **CEO question:** where is cost of goods actually concentrated?

**Formula (Opex):** Σ(amount) grouped by expense_category, as a % of
total opex. **CEO question:** where does the fixed overhead that isn't
tied to any single order actually go?

(`opex.csv` is a Stage-2 addition to the dataset -- Layer 1's "operating
expense breakdown" KPI needs overhead data that no order-level row can
ever contain, since rent/salaries/tools aren't a property of any one
order. See `src/data_generator.py::generate_opex`.)

In [18]:
cogs_breakdown = core.cogs_breakdown(orders_clean)
print("COGS breakdown by category:")
print(cogs_breakdown.round(3).to_string(index=False))

print()
opex_breakdown = core.opex_breakdown(opex_df)
print("Operating expense breakdown by category:")
print(opex_breakdown.round(3).to_string(index=False))

COGS breakdown by category:
   category   net_cogs  pct_of_total_cogs
  Outerwear 4665996.17              0.291
   Footwear 3565411.18              0.223
    Dresses 2718968.15              0.170
    Bottoms 2541071.93              0.159
       Tops 1739243.64              0.109
Accessories  781012.10              0.049

Operating expense breakdown by category:
   expense_category  total_amount  pct_of_total_opex
    Salaries & Team    6041200.73              0.561
   Rent & Utilities    1919118.29              0.178
  Warehousing & Ops    1463030.31              0.136
   Software & Tools     762534.41              0.071
Professional & Misc     585474.34              0.054


### KPI 5 — The monthly P&L: Operating Profit

**Formula:** Operating Profit = Contribution Margin − Operating Expenses.
Operating Margin % = Operating Profit / Net Revenue.
**CEO question:** after covering product, fulfillment, AND fixed
overhead, is the business actually profitable month to month? (This is
still "profit before marketing" -- marketing-loaded profitability is a
Stage 3/5 concern -- documented here so that's not mistaken for the full
picture.)

In [19]:
pnl_trend = core.company_pnl_trend(orders_clean, opex_df)
money_cols = ["contribution_margin_abs", "total_opex", "operating_profit", "net_revenue"]
pnl_display = pnl_trend.copy()
pnl_display[money_cols] = pnl_display[money_cols].round(0)
pnl_display["operating_margin_pct_display"] = (pnl_display["operating_margin_pct"] * 100).round(1)
pnl_display.drop(columns=["operating_margin_pct"])

,month,contribution_margin_abs,n_orders,contribution_margin_per_order,total_opex,operating_profit,net_revenue,operating_margin_pct_display
0,2025-07,1083812.0,894,1212.318020,873469.0,210343.0,2020627.0,10.4
1,2025-08,1396756.0,1122,1244.880348,863032.0,533724.0,2584163.0,20.7
2,2025-09,1376670.0,1140,1207.604956,835817.0,540852.0,2560233.0,21.1
3,2025-10,1445377.0,1177,1228.017562,842016.0,603360.0,2693371.0,22.4
4,2025-11,1195486.0,1926,620.709226,888511.0,306975.0,3145650.0,9.8
5,2025-12,1555036.0,1294,1201.727697,867016.0,688020.0,2890627.0,23.8
6,2026-01,1821465.0,1501,1213.501039,905884.0,915581.0,3395291.0,27.0
7,2026-02,1628825.0,1442,1129.559320,913386.0,715438.0,3033764.0,23.6
8,2026-03,1882016.0,1608,1170.408246,939222.0,942795.0,3488263.0,27.0
9,2026-04,1787708.0,1608,1111.759005,940646.0,847062.0,3324136.0,25.5


### KPI 6 — Margin by SKU and SKU concentration

**Formula (margin by SKU):** for each SKU: net_revenue, net_cogs,
gross_margin_abs/pct, contribution_margin_abs, units_sold (net of
returns), contribution_margin_per_unit.
**CEO question:** which specific products actually make money, and which
sell but quietly lose money once fulfillment cost is counted? This table
feeds Stage 5's alerts directly.

**Formula (concentration):** rank SKUs by net_revenue descending;
cumulative_share = running total ÷ total net revenue. Reports the revenue
share held by the top 5/10/20 SKUs and by the top 20% of SKUs by count.
**CEO question:** how dependent is the business on a small number of hero
products? High concentration means one SKU going out of stock or falling
out of fashion is a real revenue risk.

In [20]:
margin_by_sku = core.margin_by_sku(orders_clean)
print("Top 10 SKUs by contribution margin:")
margin_by_sku.sort_values("contribution_margin_abs", ascending=False).head(10)

Top 10 SKUs by contribution margin:


,sku_id,category,net_revenue,net_cogs,contribution_margin_abs,units_sold,gross_margin_abs,gross_margin_pct,contribution_margin_per_unit
0,OUT-013,Outerwear,785851.40,308062.44,443053.25,188,477788.96,0.607989,2356.666223
3,OUT-002,Outerwear,759538.70,297941.32,427098.56,194,461597.38,0.607734,2201.538969
4,OUT-012,Outerwear,741063.50,293923.50,415123.67,195,447140.00,0.603376,2128.839333
1,OUT-010,Outerwear,780501.26,348886.64,396318.79,188,431614.62,0.552997,2108.078670
2,OUT-001,Outerwear,763096.87,348285.42,381627.54,189,414811.45,0.543590,2019.193333
5,OUT-017,Outerwear,683653.61,276227.73,377043.60,153,407425.88,0.595954,2464.337255
6,FOO-016,Footwear,644111.50,299114.88,312783.06,192,344996.62,0.535616,1629.078437
7,OUT-016,Outerwear,626223.81,286455.95,310668.80,187,339767.86,0.542566,1661.330481
8,OUT-019,Outerwear,607792.96,270400.61,307599.28,191,337392.35,0.555111,1610.467435
13,OUT-009,Outerwear,537936.34,219286.80,292626.81,180,318649.54,0.592355,1625.704500


In [21]:
sku_ranked, concentration_summary = core.sku_concentration(orders_clean)
print("SKU revenue concentration summary:")
for k, v in concentration_summary.items():
    print(f"  {k}: {v:.1%}" if isinstance(v, float) else f"  {k}: {v}")
print()
print("Top 10 SKUs by revenue (ranked, with cumulative share):")
sku_ranked[["sku_id", "category", "net_revenue", "revenue_share", "cumulative_share"]].head(10)

SKU revenue concentration summary:
  top_5_skus_revenue_share: 10.0%
  top_10_skus_revenue_share: 18.2%
  top_20_skus_revenue_share: 31.7%
  top_20pct_skus_revenue_share: 42.3%
  n_skus: 150

Top 10 SKUs by revenue (ranked, with cumulative share):


,sku_id,category,net_revenue,revenue_share,cumulative_share
0,OUT-013,Outerwear,785851.40,0.020536,0.020536
1,OUT-010,Outerwear,780501.26,0.020396,0.040931
2,OUT-001,Outerwear,763096.87,0.019941,0.060872
3,OUT-002,Outerwear,759538.70,0.019848,0.080720
4,OUT-012,Outerwear,741063.50,0.019365,0.100086
5,OUT-017,Outerwear,683653.61,0.017865,0.117951
6,FOO-016,Footwear,644111.50,0.016832,0.134782
7,OUT-016,Outerwear,626223.81,0.016364,0.151147
8,OUT-019,Outerwear,607792.96,0.015883,0.167029
9,FOO-018,Footwear,564305.41,0.014746,0.181775


The top 10 SKUs (of 150) generate about 18% of revenue, and the top 20%
of SKUs (30 products) generate about 42% -- concentrated, but not
dangerously so; no single hero product the business is dependent on.

### KPI 7 — Inventory turns and Days of Inventory Outstanding (DIO)

**Formula:** Inventory Turns (monthly) = COGS of units sold that month ÷
Average Inventory Value that month, where Average Inventory Value =
(beginning + ending inventory) / 2, valued at each SKU's unit COGS.
Annualized Turns = monthly turns × 12. **DIO = 365 ÷ Annualized Turns.**
**CEO question:** how many times a year is inventory capital being sold
through (higher = capital working harder), and on average how many days
of stock sit in the warehouse before selling (higher DIO = cash trapped
in slow-moving inventory)?

In [22]:
inventory_turns = core.inventory_turns(inventory_df, orders_clean)
inventory_turns.round(2)

,month,cogs_sold_value,avg_inventory_value,inventory_turns_monthly,inventory_turns_annualized,days_inventory_outstanding
0,2025-07,1069220.70,2876687.22,0.37,4.46,81.83
1,2025-08,1321692.63,2946794.67,0.45,5.38,67.82
2,2025-09,1350384.23,3166400.48,0.43,5.12,71.32
3,2025-10,1400388.05,3232184.84,0.43,5.20,70.20
4,2025-11,2236231.14,2677254.30,0.84,10.02,36.42
5,2025-12,1482271.32,2787261.08,0.53,6.38,57.20
6,2026-01,1766487.06,3310085.92,0.53,6.40,57.00
7,2026-02,1642116.54,3141945.67,0.52,6.27,58.20
8,2026-03,1792348.36,3088769.46,0.58,6.96,52.42
9,2026-04,1786062.41,3446994.30,0.52,6.22,58.70


### KPI 8 — Working-capital cycle (DSO, DPO, DIO, CCC)

**⚠️ Explicit assumption, not derived from data:** this dataset has no
accounts-receivable or accounts-payable sub-ledger, so **DSO and DPO are
NOT computed from data** -- they are stated business assumptions:

- **DSO (Days Sales Outstanding) = 2 days**, assumed. A D2C brand collects
  payment upfront at checkout via a payment gateway -- there is no
  customer credit period. The 2 days represents a typical payment
  gateway settlement lag (money in hand to money in the bank), not
  customer credit terms.
- **DPO (Days Payable Outstanding) = 30 days**, assumed. Represents a
  typical negotiated net-30 supplier payment term in fashion sourcing.
  There is no payables ledger in this dataset to derive it from.
- **DIO is genuinely derived** from the inventory data above -- it's the
  one real, measured input to this formula.

**Formula:** Cash Conversion Cycle (CCC) = DSO + DIO − DPO.
**CEO question:** how many days of the business's own cash are tied up
funding one cycle of buying, holding, and selling inventory, net of how
long it can delay paying suppliers? Lower (or negative) CCC means the
business needs less of its own capital to fund growth.

In [23]:
working_capital = core.working_capital_cycle(inventory_df, orders_clean)
working_capital.round(1)

,month,days_inventory_outstanding,dso_days,dpo_days,cash_conversion_cycle
0,2025-07,81.8,2.0,30.0,53.8
1,2025-08,67.8,2.0,30.0,39.8
2,2025-09,71.3,2.0,30.0,43.3
3,2025-10,70.2,2.0,30.0,42.2
4,2025-11,36.4,2.0,30.0,8.4
5,2025-12,57.2,2.0,30.0,29.2
6,2026-01,57.0,2.0,30.0,29.0
7,2026-02,58.2,2.0,30.0,30.2
8,2026-03,52.4,2.0,30.0,24.4
9,2026-04,58.7,2.0,30.0,30.7


---
## Stage 2 summary

Built the cleaning pipeline (`src/clean.py`) and the full Universal Core
KPI engine (`src/core.py`), both dependent on nothing fashion-specific.
The **pre-flight diagnostic**, run before writing any KPI code, surfaced a
real gap: **0 of 5 marketing channels and only 3 of 150 SKUs show negative
contribution margin even after CAC is allocated** -- too thin a signal for
Stage 5's alert model, and flagged explicitly rather than quietly
built around. Cleaning was proven to catch real issues (duplicate rows,
missing costs, casing, invalid values) against an injected-messiness
sample, then run for real against the canonical data, which -- as
expected, since it's generated clean -- logged zero rows affected on
every step. Eight Universal Core KPIs were computed with real output:
revenue trend, gross margin, contribution margin (plus a returns-margin
bridge showing returns cost **29% of gross contribution margin** over the
year), COGS/opex breakdown, a monthly operating-profit P&L, margin-by-SKU
with revenue concentration (top 10 SKUs = 18% of revenue), inventory
turns/DIO, and the working-capital cycle -- with DSO and DPO explicitly
labeled as stated assumptions (not derived figures), never fabricated
from data that doesn't exist. 12 new tests (27 total across both stages)
cover the cleaning decision log, the return/variable-cost model, and every
KPI formula.

**Stopping here for review before Stage 3** (the fashion industry module
via the config boundary) -- which will also be where the loss-maker
calibration gap gets fixed, since that's exactly where CAC becomes a
first-class, reusable KPI rather than diagnostic-only code.